In [ ]:
# Solutions live in solutions/. Run from the repository root so imports and paths work.
import os, pathlib, sys
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "workshop").is_dir())
os.chdir(_root)
sys.path.insert(0, str(_root))

# Station C: quak

**Good for:** exploring and filtering large tables, with a histogram above
every column. Filters become SQL, powered by DuckDB. Works offline.
By Trevor Manz, the author of anywidget.

Docs: https://github.com/manzt/quak

In [ ]:
import ipywidgets as widgets
import numpy as np
import pandas as pd
import quak

rng = np.random.default_rng(1)
n = 200_000
cities = ["Chennai", "Delhi", "Mumbai", "Kolkata", "Bengaluru", "Hyderabad", "Pune", "Jaipur"]
weather = pd.DataFrame({
    "city": rng.choice(cities, n),
    "day": pd.Timestamp("2025-01-01") + pd.to_timedelta(rng.integers(0, 365, n), unit="D"),
    "temp_c": rng.normal(29, 5, n).round(1),
    "rain_mm": rng.gamma(0.6, 8, n).round(1),
    "humidity": rng.uniform(30, 100, n).round(0),
})
f"{len(weather):,} rows"

Click and drag on a column's histogram to filter. Click a column name to sort.

In [ ]:
table = quak.Widget(weather)
table

The filters you set are SQL. Get the query and the filtered rows back:

In [ ]:
print(table.sql)
filtered = table.data().df()
filtered.describe()

### Your task: connect it

`sql` is a synced trait. Make an `Output` that shows the number of rows and the
mean temperature per city **every time you change a filter**.

In [ ]:
city_summary = widgets.Output()


def on_filter(change=None):
    if not table.sql:  # quak sends "" while a table view is still loading
        return
    df = table.data().df()
    with city_summary:
        city_summary.clear_output(wait=True)
        print(f"{len(df):,} rows match")
        display(df.groupby("city")["temp_c"].agg(["count", "mean"]).round(2))


table.observe(on_filter, names="sql")
on_filter()
city_summary